In [1]:
#Data Preprocessing

import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('/mnt/sdc/tyh/TCRFormer/IMM25/immrep2025_for_release.tsv',delimiter='\t')

In [3]:
df = df[['peptide','tcra_cdr3','tcra_v','tcra_j',
        'tcrb_cdr3','tcrb_v','tcrb_j','label']]

columns_to_clean = ['tcra_v', 'tcra_j', 'tcrb_v', 'tcrb_j']

for col in columns_to_clean:
    if col in df.columns: 
        df[col] = df[col].str.replace(r'(?<!\d)0+(\d+)', r'\1', regex=True)

df['tcra_v'] = df['tcra_v'].str.replace('TCRAV', 'TRAV', regex=False)
df['tcra_j'] = df['tcra_j'].str.replace('TCRAJ', 'TRAJ', regex=False)
df['tcrb_v'] = df['tcrb_v'].str.replace('TCRBV', 'TRBV', regex=False)
df['tcrb_j'] = df['tcrb_j'].str.replace('TCRBJ', 'TRBJ', regex=False)
df['tcrb_v'] = df['tcrb_v'].str.replace('TRBV6-2/6-3', 'TRBV6-2', regex=False)

df.to_csv('/mnt/sdc/tyh/TCRFormer/IMM25/test.csv',index=None)

In [4]:
iedb = pd.read_csv('/mnt/sdc/tyh/TCRFormer/IMM25/iedb_positives.csv')
vdjdb = pd.read_csv('/mnt/sdc/tyh/TCRFormer/IMM25/vdjdb_positives.csv')

iedb = iedb[['Peptide','CDR3a_extended','Va','Ja','CDR3b_extended','Vb','Jb']]
vdjdb = vdjdb[['Peptide','CDR3a_extended','Va','Ja','CDR3b_extended','Vb','Jb']]
train = pd.concat([iedb,vdjdb])

train.to_csv('/mnt/sdc/tyh/TCRFormer/IMM25/train.csv',index=None)
train = train[train['Peptide'].str.len() == 9]
train.to_csv('/mnt/sdc/tyh/TCRFormer/IMM25/train_v1.0.csv',index=None)

In [ ]:
#Then extract the embeddings of alpha, beta, and epitopes based on test.csv and train.csv. Here, /mnt/sdc/TCRFormer/IMM25/train/ is taken as an example

In [5]:
#run
from model import *

In [6]:
train_csv = '/mnt/sdc/tyh/TCRFormer/IMM25/train_v1.0.csv'
base_dir = '/mnt/sdc/tyh/TCRFormer/IMM25/train_emb/'

In [7]:
run(train_csv = train_csv, base_dir = base_dir)

In [8]:
#test
device = 'cuda:0'
test_emb_path = '/mnt/sdc/TCRFormer/IMM25/test_emb/'
model_path = '/mnt/sdc/tyh/TCRFormer/IMM25/model.pt'
model = torch.load(model_path)
model = model.eval()

beta_test_emb = np.load(test_emb_path+'beta_cdr3.npy')
alpha_test_emb = np.load(test_emb_path+'alpha_cdr3.npy')
ep_test_emb = np.load(test_emb_path+'epitope.npy')
test_labels = np.load(test_emb_path+'label.npy')

test_dataset = sx_Dataset(beta_test_emb,alpha_test_emb,ep_test_emb,test_labels)
test_dataloader= DataLoader(dataset=test_dataset,batch_size=32,shuffle=False,num_workers=4,drop_last=False)

preds=[]

for te_step, (bte,ate,pep,tl) in enumerate(test_dataloader):   
    pep=torch.tensor(pep,dtype=torch.float32).to(device) 
    bte=torch.tensor(bte,dtype=torch.float32).to(device)    
    ate=torch.tensor(ate,dtype=torch.float32).to(device) 
    tl=torch.tensor(tl,dtype=torch.float32).to(device)  

    pred = model(bte,ate,pep)
    pred = pred.flatten().detach().cpu().numpy()
    
    preds.append(pred)

preds=np.concatenate(preds)

In [9]:
Presult = {k: list(v) for k, v in df.groupby('peptide').groups.items()}
for ep,index in Presult.items():
    print(ep,roc_auc_score(test_labels[index],preds[index]))

FEAQPGALL 0.6036222222222222
FLDCKSYIL 0.5217777777777778
GENALTYAL 0.38884444444444444
GEVLACYAL 0.5305777777777777
HLDDYPYLM 0.5087999999999999
ILHTHVPEV 0.4724
ILMHATYFL 0.5807555555555556
KLYPFLWFA 0.5812888888888889
MEMPDYLLL 0.5706222222222223
MENWSALEL 0.6392
MTDYDYLEV 0.4547111111111111
REDDYSVWL 0.5727111111111112
SEESAFYVL 0.5246222222222222
SQFNWTIYL 0.5981333333333334
TVYPYGTSL 0.5626888888888889
YEDGVIFYL 0.5952
YENGSTPVL 0.49937777777777775
YESYIPGAL 0.5940888888888889
YLFNADIWI 0.6645333333333333
YMFYDGYDV 0.5516444444444444
